# Разведочный Sanity Check: частоты цифр числа $\pi$, скорость `mpmath` и контроль перемешиванием

Короткий стартовый ноутбук проекта перед запуском основного пайплайна (`experiments/01..07_*.py`).
Здесь проверяются три базовые вещи:
1. Скорость генерации $100\,000$ знаков через `mpmath` на иррациональных константах ($\pi, e, \sqrt{2}$) и рациональных дробях (`1/97`, `1/999983`).
2. Равномерность распределения первых $100\,000$ цифр $\pi$ (критерий $\chi^2$).
3. Сравнение `true_pi`, случайной перестановки `np.random.permutation(pi_digits)` и периодической дроби `1/97` по автокорреляции и `LZMA`-сжатию.

## 1. Скорость генерации цифр через `mpmath`

In [ ]:
import lzma
import time

import mpmath as mp
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats

N_DIGITS = 100_000
SEED = 42


def generate_mpmath_digits(kind: str, n: int = N_DIGITS) -> tuple[np.ndarray, float]:
    t0 = time.perf_counter()
    mp.mp.dps = n + 20
    if kind == "pi":
        s = mp.nstr(mp.pi, n + 5)
    elif kind == "e":
        s = mp.nstr(mp.e, n + 5)
    elif kind == "sqrt2":
        s = mp.nstr(mp.sqrt(2), n + 5)
    elif kind == "1/97":
        s = mp.nstr(mp.mpf(1) / 97, n + 10)
    elif kind == "1/999983":
        s = mp.nstr(mp.mpf(1) / 999983, n + 15)
    else:
        raise ValueError(f"Неизвестный тип последовательности: {kind}")

    frac = s.split(".", 1)[1][:n]
    arr = np.frombuffer(frac.encode("ascii"), dtype=np.uint8) - ord("0")
    elapsed = time.perf_counter() - t0
    return arr, elapsed


sequences = {}
timings = []
for name in ["pi", "e", "sqrt2", "1/97", "1/999983"]:
    digits, dt = generate_mpmath_digits(name, N_DIGITS)
    sequences[name] = digits
    timings.append({"sequence": name, "n_digits": len(digits), "time_sec": round(dt, 4)})

pd.DataFrame(timings)

## 2. Проверка равномерности первых $100\,000$ цифр $\pi$ ($\chi^2$-тест)

In [ ]:
pi_digits = sequences["pi"]
counts = np.bincount(pi_digits, minlength=10)
expected = np.full(10, len(pi_digits) / 10.0)
chi2_stat, chi2_p = stats.chisquare(counts, f_exp=expected)

freq_df = pd.DataFrame({
    "digit": np.arange(10),
    "count": counts,
    "share_pct": np.round(100.0 * counts / len(pi_digits), 3),
})
print(f"chi2 = {chi2_stat:.4f}, p-value = {chi2_p:.4f} -> {'PASS (равномерно)' if chi2_p > 0.05 else 'FAIL'}")
display(freq_df)

fig_freq = px.bar(
    freq_df,
    x="digit",
    y="count",
    title=f"Частоты первых {N_DIGITS:,} цифр pi (chi2 p = {chi2_p:.4f})",
    template="plotly_white",
)
fig_freq.add_hline(y=len(pi_digits) / 10.0, line_dash="dash", line_color="crimson", annotation_text="Ожидаемое 10%")
fig_freq.show()

## 3. Sanity Check: истинное $\pi$ vs перемешанное $\pi$ vs периодическая дробь `1/97`

In [ ]:
rng = np.random.default_rng(SEED)
pi_shuffled = rng.permutation(pi_digits)


def max_acf_and_lzma(arr: np.ndarray, max_lag: int = 20) -> dict[str, float]:
    x = arr.astype(np.float64) - np.mean(arr)
    denom = float(np.dot(x, x))
    acfs = [abs(float(np.dot(x[:-lag], x[lag:]) / denom)) for lag in range(1, max_lag + 1)]
    raw_bytes = (arr + ord("0")).tobytes()
    lzma_bits_per_digit = 8.0 * len(lzma.compress(raw_bytes)) / len(arr)
    return {
        "max_abs_acf_lag1_20": round(max(acfs), 5),
        "lzma_bits_per_digit": round(lzma_bits_per_digit, 4),
    }


sanity_rows = []
for label, arr in [
    ("true_pi", pi_digits),
    ("shuffled_pi", pi_shuffled),
    ("rational_1_97", sequences["1/97"]),
]:
    metrics = max_acf_and_lzma(arr)
    sanity_rows.append({"sequence": label, **metrics})

sanity_df = pd.DataFrame(sanity_rows)
sanity_df

## Итоги разведочной проверки

Первые $100\,000$ десятичных знаков $\pi$ проходят тест $\chi^2$ на равномерность и по статистике сжатия `LZMA` и автокорреляции не отличаются от собственной случайной перестановки `shuffled_pi`. В то же время рациональная дробь `1/97` сразу выдаёт скрытую периодичность через низкий битрейт `LZMA` и выраженные пики автокорреляции. Для перехода к строгим экспериментам с $10^6$ знаков и нейросетями запускайте скрипты из каталога `experiments/`.